# DVWA LLM Vulnerability-Repair Pipeline — L0 / L1 / L2

Repair model: **Qwen2.5-14B-Instruct**, 4-bit NF4, greedy decoding, 4096-token budget.

| Level | Context supplied to the model |
|---|---|
| **L0** | Raw PHP source only |
| **L1** | Source + SAST findings + application context (where available) |
| **L2** | Source + SAST + DAST + application context + repair guidance |

Outcomes are `CFR` (correct fix), `PIR` (plausible but incorrect) and `IFR` (invalid fix),
as defined in CELL 7.

**Run order.** CELLS 1-17 are the cascade and require a GPU at CELLS 3, 9, 12 and 15.
CELLS 18-24 are the post-hoc validation and analysis; only CELL 20 needs a GPU.
To reproduce the reported numbers from saved outputs: CELL 1, CELL 2, CELL 7,
CELL 18, CELL 19, CELL 23, CELL 24.


In [ ]:
# CELL 1 — Install dependencies + mount Google Drive
import subprocess, sys

print("Installing Python packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "transformers", "accelerate", "bitsandbytes", "semgrep", "-q"],
    check=True,
)

print("Installing php-cli for syntax checking...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "php-cli"], check=True)

from google.colab import drive
drive.mount("/content/drive")
print("\nSetup complete.")

In [ ]:
# CELL 2 — Imports & path configuration
import os, re, json, time, subprocess, tempfile
from pathlib import Path
from datetime import datetime, timezone

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

REPO_ROOT   = Path("/content/drive/MyDrive/dasttt")
DATASET_DIR = REPO_ROOT / "dvwa_selected_v2"
SAST_JSON   = REPO_ROOT / "sast_normalized.json"
DAST_JSON   = REPO_ROOT / "dast_report.json"

L0_OUTPUT_DIR  = REPO_ROOT / "dvwa_v3_l0_outputs"
L0_RESULTS_DIR = REPO_ROOT / "dvwa_v3_l0_results"
L1_OUTPUT_DIR  = REPO_ROOT / "dvwa_v3_l1_outputs"
L1_RESULTS_DIR = REPO_ROOT / "dvwa_v3_l1_results"
L2_OUTPUT_DIR  = REPO_ROOT / "dvwa_v3_l2_outputs"
L2_RESULTS_DIR = REPO_ROOT / "dvwa_v3_l2_results"

for _d in [L0_OUTPUT_DIR, L0_RESULTS_DIR, L1_OUTPUT_DIR, L1_RESULTS_DIR,
           L2_OUTPUT_DIR, L2_RESULTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

# No modules are excluded at load time.
EXCLUDED_VULN_DIRS = set()

VULN_CWE_MAP = {
    "sqli":          "89",
    "sqli_blind":    "89",
    "xss_r":         "79",
    "xss_s":         "79",
    "xss_d":         "79",
    "exec":          "78",
    "fi":            "22",
    "upload":        "434",
    "csrf":          "352",
    "brute":         "307",
    "weak_id":       "330",
    "open_redirect": "601",
    "captcha":       "287",
    "csp":           "693",
    "api":           "78",
    "authbypass":    "284",
    "bac":           "89",
    "javascript":    "602",
}

# Application context injected into L1 and L2 prompts.
# Per-case keys ({vuln_dir}_{stem}) take priority over module keys ({vuln_dir}).
VULN_FRONTEND_CONTEXT = {
    # javascript (CWE-602): per-case, because each level uses different client-side logic
    "javascript_low": (
        "Framework context: This file injects client-side JavaScript that generates a token "
        "via md5(rot13(phrase)) and puts it in a hidden <input id='token'>. "
        "The form POSTs 'phrase' and 'token'. The server currently validates BOTH "
        "phrase=='success' AND token==md5(str_rot13('success')). "
        "The vulnerability (CWE-602) is that security relies on client-side JS which can be "
        "inspected and bypassed. "
        "Fix: Replace the obfuscated MD5+rot13 JavaScript with PHP that validates $_POST['phrase'] "
        "directly server-side: if($_POST['phrase'] !== 'success') reject the request."
    ),
    "javascript_medium": (
        "Framework context: This file loads medium.js which generates the token by reversing "
        "'XX' + phrase + 'XX'. Concretely: token = reverse('XX' + phrase + 'XX'). "
        "The server checks token == strrev('XXsuccessXX'). "
        "The vulnerability (CWE-602) is that security relies on client-side JS which can be "
        "analyzed to compute the expected token for any phrase. "
        "Fix: Remove the <script> include. Add PHP server-side validation: "
        "check $_POST['phrase'] === 'success' directly. "
        "You may also add a server-side token computed as strrev('XX' . $_POST['phrase'] . 'XX')."
    ),
    "javascript_high": (
        "Framework context: This file loads high.js which contains an obfuscated sha256 library. "
        "The token computation is: sha256(sha256('XX' + reverse(phrase)) + 'ZZ'). "
        "The server checks: hash('sha256', hash('sha256', 'XX' . strrev('success')) . 'ZZ'). "
        "The vulnerability (CWE-602) is that security relies on client-side sha256 obfuscation "
        "which can be reverse-engineered to compute valid tokens. "
        "Fix: Remove the <script> include. Add PHP server-side validation: "
        "check $_POST['phrase'] === 'success' and compute the token server-side using "
        "hash('sha256', hash('sha256', 'XX' . strrev($_POST['phrase'])) . 'ZZ') for comparison."
    ),
    # xss_d (CWE-79, DOM-based): the repair is a server-side whitelist
    "xss_d": (
        "Framework context: This page renders a language selector. The vulnerable JavaScript in "
        "index.php does: var lang = document.location.href.substring(indexOf('default=')+8); "
        "document.write('<option value=' + lang + '>' + decodeURI(lang) + '</option>'); "
        "This is DOM-based XSS — user-controlled URL fragment is written to the DOM without sanitization. "
        "The PHP source file receives the GET parameter: $_GET['default']. "
        "Fix: Add server-side PHP whitelist validation. Only allow: English, French, Spanish, German. "
        "Use a switch/case or in_array() check. If the value is not in the whitelist, "
        "redirect to '?default=English' and exit. This prevents XSS payloads from reaching the DOM."
    ),
    "csrf": (
        "Framework context: This is a password-change form (GET params: password_new, password_conf, Change). "
        "The DVWA framework provides two helper functions for CSRF protection:\n"
        "  - generateSessionToken(): stores a token in $_SESSION['session_token'] and outputs a "
        "hidden <input name='user_token'> field.\n"
        "  - checkToken($request_token, $session_token, $redirect): validates the submitted token "
        "against the session token; redirects to $redirect on failure.\n"
        "The fix must call checkToken() at the start of the handler and generateSessionToken() at the end."
    ),
    "authbypass": (
        "Framework context: This PHP file guards an admin-only user management panel. "
        "The page title is 'Vulnerability: Authorisation Bypass' and the HTML states: "
        "'This page should only be accessible by the admin user.' "
        "The DVWA framework provides dvwaCurrentUser() which returns the username of the "
        "currently authenticated user. "
        "If the current user is not 'admin', the file must immediately print 'Unauthorised', "
        "call http_response_code(403), and exit."
    ),
    "captcha": (
        "Framework context: This file handles a password-change form protected by Google reCAPTCHA "
        "(Insecure CAPTCHA challenge). The vulnerability is a two-step bypass: step=1 validates "
        "the CAPTCHA, but step=2 changes the password WITHOUT re-validating. "
        "The fix must ensure the CAPTCHA is verified on every password change attempt — "
        "either by removing the two-step structure or by re-validating in step=2. "
        "Use recaptcha_check_answer($_DVWA['recaptcha_private_key'], $_POST['g-recaptcha-response']) "
        "which is available via the already-included recaptchalib.php."
    ),
}

# Vulnerability-specific repair guidance, added only to the L2 prompt.
# Per-case keys ({vuln_dir}_{stem}) take priority over module keys ({vuln_dir}).
VULN_L2_HINTS = {
    "javascript_low": (
        "Fix instruction: Remove all the MD5/rot13 JavaScript. Replace with PHP server-side validation:\n"
        "  if (isset($_POST['phrase']) && isset($_POST['token'])) {\n"
        "    if ($_POST['phrase'] !== 'success') {\n"
        "      $html .= '<pre>Wrong phrase. Try again.</pre>';\n"
        "    } else {\n"
        "      $html .= '<pre>Well done!</pre>';\n"
        "    }\n"
        "  }\n"
        "Also output a hidden token computed server-side: echo md5(str_rot13('success'));"
    ),
    "javascript_medium": (
        "Fix instruction: Remove the <script src='medium.js'> include. Replace with PHP:\n"
        "  if (isset($_POST['phrase'])) {\n"
        "    $expected_token = strrev('XX' . $_POST['phrase'] . 'XX');\n"
        "    if ($_POST['phrase'] !== 'success' || $_POST['token'] !== $expected_token) {\n"
        "      $html .= '<pre>Invalid.</pre>';\n"
        "    } else { $html .= '<pre>Well done!</pre>'; }\n"
        "  }\n"
        "Compute the token server-side so no JS is needed for security."
    ),
    "javascript_high": (
        "Fix instruction: Remove the <script src='high.js'> include. Replace with PHP:\n"
        "  if (isset($_POST['phrase'])) {\n"
        "    $expected = hash('sha256', hash('sha256', 'XX' . strrev($_POST['phrase'])) . 'ZZ');\n"
        "    if ($_POST['phrase'] !== 'success' || $_POST['token'] !== $expected) {\n"
        "      $html .= '<pre>Invalid.</pre>';\n"
        "    } else { $html .= '<pre>Well done!</pre>'; }\n"
        "  }"
    ),
    "xss_d_low": (
        "Fix instruction: The source file is currently empty (just a PHP comment). "
        "Add a server-side whitelist for the 'default' GET parameter:\n"
        "  if (array_key_exists('default', $_GET)) {\n"
        "    switch ($_GET['default']) {\n"
        "      case 'French': case 'English': case 'German': case 'Spanish': break;\n"
        "      default: header('location: ?default=English'); exit;\n"
        "    }\n"
        "  }"
    ),
    "xss_d_medium": (
        "Fix instruction: The current code strips <script> tags but DOM XSS does not need script tags. "
        "Replace with a proper whitelist:\n"
        "  if (array_key_exists('default', $_GET)) {\n"
        "    switch ($_GET['default']) {\n"
        "      case 'French': case 'English': case 'German': case 'Spanish': break;\n"
        "      default: header('location: ?default=English'); exit;\n"
        "    }\n"
        "  }"
    ),
    "open_redirect": (
        "Fix instruction: Validate the redirect parameter with is_numeric() first. "
        "Then use a switch/case that maps each allowed integer ID to a hardcoded target URL. "
        "Never pass $_GET values directly to header('Location: ...'). "
        "If the value is not in the whitelist, display an error and do not redirect."
    ),
    "upload": (
        "Fix instruction: Verify the uploaded file is a real image using getimagesize() on the "
        "temp file path. Only accept jpg/jpeg/png — check both the extension (strtolower + comparison) "
        "and the MIME type ($_FILES['uploaded']['type']). Reject all other files before moving them."
    ),
    "weak_id": (
        "Fix instruction: Generate the cookie value with strong randomness: "
        "sha1(mt_rand() . time() . 'ImpossibleSalt') or bin2hex(random_bytes(32)). "
        "Pass both secure=true and httponly=true to setcookie(). "
        "Never use sequential integers or predictable values as session identifiers."
    ),
    "csrf": (
        "Fix instruction: At the very top of the if(isset($_GET['Change'])) block, call "
        "checkToken($_REQUEST['user_token'], $_SESSION['session_token'], 'index.php'). "
        "At the very bottom of the file (outside all if blocks), call generateSessionToken(). "
        "These are DVWA framework functions — do not reimplement them."
    ),
    "authbypass": (
        "Fix instruction: The first lines of PHP (after <?php) must be:\n"
        "  if (dvwaCurrentUser() != 'admin') {\n"
        "      print 'Unauthorised';\n"
        "      http_response_code(403);\n"
        "      exit;\n"
        "  }\n"
        "dvwaCurrentUser() is a DVWA framework function. This is the complete fix."
    ),
    "captcha": (
        "Fix instruction: Remove the two-step structure entirely. In a single handler, "
        "call recaptcha_check_answer($_DVWA['recaptcha_private_key'], $_POST['g-recaptcha-response']). "
        "If CAPTCHA fails, show an error and stop. If it passes, update the password. "
        "Do not have a step=2 path that skips CAPTCHA validation."
    ),
}

print("Configuration complete.")
print(f"  Excluded vuln dirs : {EXCLUDED_VULN_DIRS}")
print(f"  Frontend context   : {set(VULN_FRONTEND_CONTEXT.keys())}")
print(f"  L2 hints           : {set(VULN_L2_HINTS.keys())}")

In [ ]:
# CELL 3 — Load Qwen2.5-14B-Instruct (4-bit quantised for T4 / A100)
print(f"Loading {MODEL_ID} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")

In [ ]:
# CELL 4 — Load & index SAST data
if not SAST_JSON.exists():
    raise FileNotFoundError(f"SAST JSON not found: {SAST_JSON}")

raw_sast = json.loads(SAST_JSON.read_text())
SAST_INDEX = {}

for _f in raw_sast:
    fpath = _f.get("file", "")
    if "vulnerabilities/" in fpath:
        rel = fpath.split("vulnerabilities/")[-1]
    else:
        continue
    SAST_INDEX.setdefault(rel, []).append(_f)

print(f"SAST index built: {len(SAST_INDEX)} file paths, {len(raw_sast)} total findings")
for k in sorted(SAST_INDEX.keys()):
    print(f"  {k}: {len(SAST_INDEX[k])} finding(s)")

In [ ]:
# CELL 5 — Load & index DAST data
if not DAST_JSON.exists():
    raise FileNotFoundError(f"DAST JSON not found: {DAST_JSON}")

raw_dast = json.loads(DAST_JSON.read_text())
DAST_ZAP_INDEX  = {}
DAST_FFUF_INDEX = {}
_vuln_path_re = re.compile(r"/vulnerabilities/([^/]+)/")

for _entry in raw_dast:
    _tool     = _entry.get("tool", "")
    _endpoint = _entry.get("endpoint", "")
    _m = _vuln_path_re.search(_endpoint)
    if not _m:
        continue
    _folder = _m.group(1)
    if _tool == "owasp-zap":
        DAST_ZAP_INDEX.setdefault(_folder, []).append(_entry)
    elif _tool == "ffuf":
        DAST_FFUF_INDEX.setdefault(_folder, []).append(_entry)

_zap_total  = sum(len(v) for v in DAST_ZAP_INDEX.values())
_ffuf_total = sum(len(v) for v in DAST_FFUF_INDEX.values())
print(f"DAST index built: {_zap_total} ZAP findings, {_ffuf_total} FFUF endpoints")
for k in sorted(DAST_ZAP_INDEX.keys()):
    cwes = sorted({str(e.get("cweid", "?")) for e in DAST_ZAP_INDEX[k]})
    print(f"  {k:<22}: {len(DAST_ZAP_INDEX[k])} ZAP findings  CWEs={cwes}")

In [ ]:
# CELL 6 — Load DVWA cases
# impossible.php is skipped: it is DVWA's reference implementation, not a test case.

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"DATASET_DIR not found: {DATASET_DIR}")

ALL_CASES = []

for _vuln_dir in sorted(DATASET_DIR.iterdir()):
    if not _vuln_dir.is_dir():
        continue
    if _vuln_dir.name in EXCLUDED_VULN_DIRS:
        print(f"  SKIP (excluded): {_vuln_dir.name}")
        continue
    _src_dir = _vuln_dir / "source"
    if not _src_dir.exists():
        continue
    _folder_cwe = VULN_CWE_MAP.get(_vuln_dir.name, "0")

    for _src_file in sorted(_src_dir.glob("*.php")):
        if _src_file.stem == "impossible":
            continue  # already-fixed reference — not a test case

        _rel_path      = f"{_vuln_dir.name}/source/{_src_file.name}"
        _sast_findings = SAST_INDEX.get(_rel_path, [])
        _file_cwe      = str(_sast_findings[0]["cwe"]) if _sast_findings else _folder_cwe

        # Per-case key takes priority over folder key for frontend context
        _case_key = f"{_vuln_dir.name}_{_src_file.stem}"
        _frontend_ctx = VULN_FRONTEND_CONTEXT.get(_case_key, VULN_FRONTEND_CONTEXT.get(_vuln_dir.name, ""))

        ALL_CASES.append({
            "case_id":          f"{_vuln_dir.name}_{_src_file.stem}",
            "vuln_dir":         _vuln_dir.name,
            "filename":         _src_file.name,
            "cwe_id":           _file_cwe,
            "rel_path":         _rel_path,
            "source":           _src_file.read_text(errors="replace"),
            "sast_findings":    _sast_findings,
            "dast_zap":         DAST_ZAP_INDEX.get(_vuln_dir.name, []),
            "dast_ffuf":        DAST_FFUF_INDEX.get(_vuln_dir.name, []),
            "frontend_context": _frontend_ctx,
        })

print(f"\nTotal cases loaded: {len(ALL_CASES)}")
print(f"  (impossible.php excluded — reference)")
print(f"\n  {'Vuln folder':<25} {'Files':>5} {'CWE':>5} {'SAST':>6} {'DAST ZAP':>9} {'Has context':>12}")
print("  " + "-" * 65)
for _vd in sorted(set(c["vuln_dir"] for c in ALL_CASES)):
    _vc    = [c for c in ALL_CASES if c["vuln_dir"] == _vd]
    _sn    = sum(len(c["sast_findings"]) for c in _vc)
    _dn    = len(_vc[0]["dast_zap"])
    _ctx   = "yes" if any(c["frontend_context"] for c in _vc) else ""
    print(f"  {_vd:<25} {len(_vc):>5} {_vc[0]['cwe_id']:>5} {_sn:>6} {_dn:>9} {_ctx:>12}")

In [ ]:
# CELL 7 — PHP lint, Semgrep rescan, code extraction, structural verifiers, classifier

def php_syntax_ok(code):
    if not code.strip():
        return False, "Empty code"
    with tempfile.NamedTemporaryFile(mode="w", suffix=".php", delete=False) as _f:
        _f.write(code)
        _tmp = _f.name
    try:
        _r = subprocess.run(["php", "-l", _tmp], capture_output=True, text=True, timeout=15)
        _ok = _r.returncode == 0
        return _ok, "" if _ok else _r.stderr[:300]
    except FileNotFoundError:
        return True, "php not installed — skipping lint"
    finally:
        os.unlink(_tmp)


def run_semgrep_php(source, filename="test.php"):
    if not source.strip():
        return []
    with tempfile.NamedTemporaryFile(mode="w", suffix=".php", delete=False) as _f:
        _f.write(source)
        _tmp = _f.name
    try:
        _r = subprocess.run(
            ["semgrep", "--config", "p/php", "--json", "--quiet", _tmp],
            capture_output=True, text=True, timeout=120,
        )
        _out = json.loads(_r.stdout) if _r.stdout.strip() else {"results": []}
        _findings = []
        for _res in _out.get("results", []):
            _cwe_list = _res.get("extra", {}).get("metadata", {}).get("cwe", [])
            if isinstance(_cwe_list, str):
                _cwe_list = [_cwe_list]
            _cwe_nums = [
                re.search(r"(\d+)", _c).group(1)
                for _c in _cwe_list if re.search(r"\d+", _c)
            ]
            _findings.append({
                "line":    _res["start"]["line"],
                "rule_id": _res["check_id"],
                "message": _res["extra"].get("message", ""),
                "cwes":    _cwe_nums,
            })
        return _findings
    except Exception as _e:
        print(f"  [Semgrep error] {_e}")
        return []
    finally:
        os.unlink(_tmp)


def extract_code(response):
    if not response:
        return ""
    _fenced = re.search(r"```(?:php)?\n(.*?)```", response, re.DOTALL)
    if _fenced:
        return _fenced.group(1).strip()
    _lines = response.strip().splitlines()
    if _lines and _lines[0].startswith("```"):
        _lines = _lines[1:]
    if _lines and _lines[-1].strip() == "```":
        _lines = _lines[:-1]
    return "\n".join(_lines)


# ── Custom structural verifiers ───────────────────────────────────────────
# Used when Semgrep reports no finding for the target CWE on the original file.
# (original, fix) -> (is_fixed: bool, reason: str)

def _verify_csrf(original, fix):
    # Accept DVWA helper functions OR generic $_SESSION token patterns
    has_check = "checkToken(" in fix or bool(re.search(
        r"\$_SESSION\s*\[.*token.*\]\s*(==|===)\s*\$_(POST|GET|REQUEST|SERVER)", fix, re.IGNORECASE
    ))
    has_gen = "generateSessionToken(" in fix or bool(re.search(
        r"\$_SESSION\s*\[.*token.*\]\s*=\s*(md5|sha1|bin2hex|hash|uniqid|random_bytes)\s*\(", fix, re.IGNORECASE
    ))
    if has_check and has_gen:
        return True, "CSRF token validation + generation present"
    missing = []
    if not has_check: missing.append("token validation")
    if not has_gen:   missing.append("token generation")
    return False, f"Missing CSRF controls: {', '.join(missing)}"


def _verify_xss(original, fix):
    if re.search(r"htmlspecialchars\s*\(", fix) or re.search(r"htmlentities\s*\(", fix):
        return True, "Output sanitized with htmlspecialchars/htmlentities"
    return False, "No htmlspecialchars/htmlentities on user output"


def _verify_fi(original, fix):
    direct_include = bool(re.search(
        r"(include|require)(_once)?\s*\(\s*\$_(GET|POST|REQUEST)", fix, re.IGNORECASE
    ))
    if direct_include:
        return False, "include/require still uses raw user input directly"
    if re.search(r"in_array\s*\(", fix):
        return True, "File inclusion protected with in_array() whitelist"
    if "basename(" in fix or "realpath(" in fix:
        return True, "File path sanitized with basename()/realpath()"
    if not re.search(r"(include|require)", fix, re.IGNORECASE):
        return False, "No include/require found — fix may have removed functionality"
    return False, "No whitelist (in_array) or path sanitization (basename/realpath) found"


def _verify_upload(original, fix):
    if "getimagesize(" in fix:
        return True, "File type verified with getimagesize()"
    has_ext = bool(re.search(r"strtolower.*\.(jpg|jpeg|png)|'jpg'|'jpeg'|'png'", fix, re.IGNORECASE))
    has_mime = bool(re.search(r"image/(jpeg|png|jpg)", fix))
    if has_ext and has_mime:
        return True, "File type verified with extension + MIME type whitelist"
    return False, "Missing server-side file type check (getimagesize or ext+MIME whitelist)"


def _verify_weak_id(original, fix):
    for _pat in [r"random_bytes\s*\(", r"openssl_random_pseudo_bytes\s*\(",
                 r"sha1\s*\(.*mt_rand|mt_rand.*sha1", r"hash\s*\(", r"bin2hex\s*\("]:
        if re.search(_pat, fix):
            return True, "Strong unpredictable ID generation found"
    if re.search(r"setcookie", fix) and re.search(r"sha1|md5|hash", fix) and \
       re.search(r"true\s*,\s*true", fix):
        return True, "Cookie set with hashed value + secure + httponly flags"
    return False, "No strong random ID generation (random_bytes / sha1(mt_rand()) / hash)"


def _verify_open_redirect(original, fix):
    direct_redirect = bool(re.search(
        r'header\s*\(\s*["\']location:.*\$_(GET|POST|REQUEST)', fix, re.IGNORECASE | re.DOTALL
    ))
    if direct_redirect:
        return False, "header(location:) still passes raw user input directly"
    if "is_numeric(" in fix and ("switch" in fix or ("case " in fix and "break" in fix)):
        return True, "Redirect uses is_numeric() + whitelist switch/case"
    if "is_numeric(" in fix:
        return True, "Redirect validates input with is_numeric()"
    if "header" not in fix.lower():
        return False, "No redirect logic found in fix"
    return False, "Redirect target not validated with is_numeric() + whitelist"


def _verify_captcha(original, fix):
    # Requires server-side CAPTCHA verification, and no step=2 branch that
    # changes the password without re-validating it.
    has_captcha = bool(re.search(
        r"recaptcha_check_answer|g-recaptcha-response|\$resp\b", fix
    ))
    if not has_captcha:
        return False, "No server-side CAPTCHA verification (recaptcha_check_answer) found"
    # Check for step=2 that skips captcha (the original bypass pattern)
    step2_block = re.search(r"step.*==.*['\"]2['\"]", fix, re.IGNORECASE)
    if step2_block:
        step2_has_captcha = bool(re.search(
            r"step.*==.*['\"]2['\"].*recaptcha_check_answer|recaptcha_check_answer.*step.*==.*['\"]2['\"]",
            fix, re.IGNORECASE | re.DOTALL
        ))
        if not step2_has_captcha:
            return False, "Two-step bypass still present: step=2 skips CAPTCHA validation"
    return True, "Server-side CAPTCHA verification present without step-2 bypass"


def _verify_authbypass(original, fix):
    has_user_check = bool(re.search(r"dvwaCurrentUser\s*\(\s*\)", fix))
    has_deny = bool(re.search(r"403|Unauthori[sz]ed|forbidden|exit\s*;", fix, re.IGNORECASE))
    if has_user_check and has_deny:
        return True, "Auth check on current user + deny response (403/exit) present"
    if has_user_check:
        return True, "Auth check on current user present"
    return False, "No server-side authorization check (dvwaCurrentUser) found"


def _verify_javascript(original, fix):
    # The repair must add server-side PHP validation, not only client-side JS.
    has_post_access = bool(re.search(r"\$_(POST|GET|REQUEST)\s*\[", fix))
    has_comparison = bool(re.search(
        r"(===?|!==?|preg_match|filter_var|in_array|strcmp)\s*.*\$_(POST|GET|REQUEST)|"
        r"\$_(POST|GET|REQUEST).*\s*(===?|!==?)",
        fix, re.IGNORECASE
    ))
    has_php_logic = bool(re.search(r"if\s*\(", fix)) and ("$" in fix)
    orig_has_script = bool(re.search(r"<script|script\s+src", original, re.IGNORECASE))
    fix_has_script = bool(re.search(r"<script|script\s+src", fix, re.IGNORECASE))
    script_removed = orig_has_script and not fix_has_script
    if has_post_access and has_comparison:
        return True, "Server-side PHP validation of POST input present"
    if has_post_access and has_php_logic:
        return True, "Server-side PHP logic with form input access present"
    if script_removed and has_php_logic:
        return True, "Client-side JS removed; server-side PHP logic added"
    return False, "No server-side PHP validation found (still JS-only or empty)"


def _verify_xss_d(original, fix):
    # A server-side whitelist is the reference repair for DOM-based XSS.
    has_whitelist = bool(re.search(
        r"(switch\s*\(\s*\$_(GET|POST|REQUEST)|in_array\s*\(\s*\$_(GET|POST|REQUEST))",
        fix, re.IGNORECASE
    ))
    has_language_check = bool(re.search(
        r"(English|French|German|Spanish)", fix
    ))
    has_redirect = bool(re.search(r"header\s*\(\s*['\"]location", fix, re.IGNORECASE))
    has_sanitization = bool(re.search(r"(htmlspecialchars|htmlentities|strip_tags|filter_var)", fix))
    has_js_safe = bool(re.search(r"(textContent|DOMPurify|encodeURIComponent)", fix))
    if has_whitelist and has_language_check:
        return True, "Server-side language whitelist present"
    if has_whitelist and has_redirect:
        return True, "Server-side whitelist with redirect on invalid input present"
    if has_sanitization:
        return True, "Server-side HTML sanitization present"
    if has_js_safe:
        return True, "DOM-safe JavaScript output handling present"
    return False, "No DOM XSS protection found (whitelist, sanitization, or textContent)"


CUSTOM_VERIFIERS = {
    "csrf":          _verify_csrf,
    "fi":            _verify_fi,
    "xss_r":         _verify_xss,
    "xss_s":         _verify_xss,
    "xss_d":         _verify_xss_d,
    "javascript":    _verify_javascript,
    "upload":        _verify_upload,
    "weak_id":       _verify_weak_id,
    "open_redirect": _verify_open_redirect,
    "captcha":       _verify_captcha,
    "authbypass":    _verify_authbypass,
}


def classify_dvwa(original, fix, filename, known_cwe, vuln_dir=None):
    """CFR / PIR / IFR classifier.

    Stage 0: IFR pre-checks. Stage 1: php -l. Stage 2: Semgrep rescan, falling
    back to a custom verifier, then to an unverified default.
    The deciding branch is stamped into the reason string and read by CELL 24.
    """
    _base = {"orig_cwes": [], "fix_cwes": [], "new_cwes": []}

    if not fix.strip():
        return {"outcome": "IFR", "reason": "Empty output", **_base}
    if fix.strip() == original.strip():
        return {"outcome": "IFR", "reason": "Identical to original", **_base}
    if len(fix.strip()) < len(original.strip()) * 0.30:
        return {"outcome": "IFR", "reason": "Code body deleted (< 30% original size)", **_base}

    _ok, _err = php_syntax_ok(fix)
    if not _ok:
        return {"outcome": "IFR", "reason": f"PHP syntax error: {_err}", **_base}

    _orig_findings = run_semgrep_php(original, filename)
    _fix_findings  = run_semgrep_php(fix, filename)
    _orig_cwes = {c for f in _orig_findings for c in f["cwes"]}
    _fix_cwes  = {c for f in _fix_findings  for c in f["cwes"]}
    _new_cwes  = _fix_cwes - _orig_cwes

    if known_cwe in _orig_cwes:
        _known_resolved   = known_cwe not in _fix_cwes
        _resolution_label = "semgrep"
        _verify_detail    = ""
    elif vuln_dir and vuln_dir in CUSTOM_VERIFIERS:
        _known_resolved, _verify_detail = CUSTOM_VERIFIERS[vuln_dir](original, fix)
        _resolution_label = f"pattern:{vuln_dir}"
    else:
        _known_resolved   = True
        _resolution_label = "semgrep_blind_unverified"
        _verify_detail    = ""

    if _known_resolved and not _new_cwes:
        _reason = f"PHP lint OK + CWE-{known_cwe} resolved [{_resolution_label}]"
        if _verify_detail:
            _reason += f" — {_verify_detail}"
        return {"outcome": "CFR", "reason": _reason,
                "orig_cwes": sorted(_orig_cwes), "fix_cwes": sorted(_fix_cwes), "new_cwes": []}
    elif _new_cwes:
        return {"outcome": "PIR", "reason": f"New CWEs introduced: {sorted(_new_cwes)}",
                "orig_cwes": sorted(_orig_cwes), "fix_cwes": sorted(_fix_cwes),
                "new_cwes": sorted(_new_cwes)}
    else:
        _reason = f"CWE-{known_cwe} not resolved [{_resolution_label}]"
        if _verify_detail:
            _reason += f" — {_verify_detail}"
        elif _resolution_label == "semgrep":
            _reason += " — still flagged by Semgrep"
        return {"outcome": "PIR", "reason": _reason,
                "orig_cwes": sorted(_orig_cwes), "fix_cwes": sorted(_fix_cwes), "new_cwes": []}


def _print_level_table(results):
    print(f"\n  {'Vuln folder':<25} {'N':>3} {'CFR':>4} {'PIR':>4} {'IFR':>4}")
    print("  " + "-" * 42)
    for _vd in sorted(set(r["vuln_dir"] for r in results)):
        _vr  = [r for r in results if r["vuln_dir"] == _vd]
        _n, _cfr = len(_vr), sum(1 for r in _vr if r["outcome"] == "CFR")
        _pir, _ifr = sum(1 for r in _vr if r["outcome"] == "PIR"), sum(1 for r in _vr if r["outcome"] == "IFR")
        print(f"  {_vd:<25} {_n:>3} {_cfr:>4} {_pir:>4} {_ifr:>4}")


print("Helper functions defined.")
print(f"Custom verifiers registered: {sorted(CUSTOM_VERIFIERS.keys())}")

In [ ]:
# CELL 8 — Prompt builders + LLM client

def format_sast_for_prompt(findings):
    if not findings:
        return "No issues detected by static analysis."
    _lines = ["Static Analysis Findings (Semgrep):"]
    for _f in findings:
        _lines.append(
            f"  Line {_f.get('line','?')}: [{_f.get('vulnerability','')}] "
            f"{_f.get('why','')}  (CWE-{_f.get('cwe','N/A')}, Severity: {_f.get('severity','')})"
        )
    return "\n".join(_lines)


def format_dast_for_prompt(zap_findings, ffuf_findings):
    _lines = ["Dynamic Analysis Findings (DAST):"]
    if zap_findings:
        _lines.append("\n  [OWASP ZAP Active Scan]")
        _seen = set()
        for _f in zap_findings:
            _key = (_f.get("vulnerability", ""), str(_f.get("cweid", "")))
            if _key in _seen:
                continue
            _seen.add(_key)
            _lines.append(
                f"    - [{_f.get('severity','?').upper()}] {_f.get('vulnerability','')}  "
                f"(CWE-{_f.get('cweid','?')})  param={_f.get('param','')!r}"
            )
    if ffuf_findings:
        _lines.append("\n  [FFUF Endpoint Discovery]")
        for _ep in [_f.get("endpoint", "") for _f in ffuf_findings]:
            _lines.append(f"    - {_ep}")
    return "\n".join(_lines) if len(_lines) > 1 else "No DAST findings."


def build_l0_prompt(source_code):
    return (
        "### Role:\n"
        "You are a secure code repair assistant specializing in fixing vulnerabilities "
        "in web application source code.\n\n"
        "### Context:\n\n"
        "#### Source Code:\n"
        f"{source_code}\n\n"
        "### Task:\n"
        "Fix all security vulnerabilities in the code above.\n\n"
        "Requirements:\n"
        "- Preserve the original functionality of the application.\n"
        "- Modify only the parts necessary to fix the vulnerabilities.\n"
        "- Do not introduce new vulnerabilities.\n"
        "- Ensure the code remains syntactically valid PHP.\n\n"
        "### Output:\n"
        "Return only the fully corrected source code. No explanations or comments."
    )


def build_l1_prompt(source_code, sast_output, frontend_context=""):
    _ctx_block = ""
    if frontend_context:
        _ctx_block = f"\n#### Application Context:\n{frontend_context}\n"
    return (
        "### Role:\n"
        "You are a secure code repair assistant specializing in fixing vulnerabilities "
        "in web application source code.\n\n"
        "### Context:\n\n"
        "#### Source Code:\n"
        f"{source_code}\n\n"
        "#### Static Analysis Report (Semgrep):\n"
        f"{sast_output}\n"
        f"{_ctx_block}\n"
        "### Task:\n"
        "Fix the vulnerabilities identified in the static analysis report.\n\n"
        "Requirements:\n"
        "- Preserve original functionality.\n"
        "- Use the SAST report to locate the exact lines and vulnerability types.\n"
        "- Do not introduce new vulnerabilities.\n"
        "- Ensure the code remains syntactically valid PHP.\n\n"
        "### Output:\n"
        "Return only the fully corrected source code."
    )


def build_l2_prompt(source_code, sast_output, dast_output, frontend_context="", vuln_hint=""):
    _ctx_block = ""
    if frontend_context:
        _ctx_block = f"\n#### Application Context:\n{frontend_context}\n"
    _hint_block = ""
    if vuln_hint:
        _hint_block = f"\n#### Specific Fix Guidance:\n{vuln_hint}\n"
    return (
        "### Role:\n"
        "You are a secure code repair assistant specializing in fixing vulnerabilities "
        "in web application source code.\n\n"
        "### Context:\n\n"
        "#### Source Code:\n"
        f"{source_code}\n\n"
        "#### Static Analysis Report (Semgrep):\n"
        f"{sast_output}\n\n"
        "#### Dynamic Analysis Report (DAST):\n"
        f"{dast_output}\n"
        f"{_ctx_block}"
        f"{_hint_block}\n"
        "### Task:\n"
        "Fix the vulnerabilities identified by both static and dynamic analysis. "
        "The DAST report confirms these vulnerabilities are exploitable at runtime.\n\n"
        "Requirements:\n"
        "- Preserve original functionality.\n"
        "- Address all findings from SAST and DAST.\n"
        "- Follow the specific fix guidance if provided.\n"
        "- Do not introduce new vulnerabilities.\n"
        "- Ensure the code remains syntactically valid PHP.\n\n"
        "### Output:\n"
        "Return only the fully corrected source code."
    )


@torch.inference_mode()
def call_model(prompt, max_new_tokens=4096):
    _messages = [
        {"role": "system", "content": "You are a secure code repair assistant."},
        {"role": "user",   "content": prompt},
    ]
    _text = tokenizer.apply_chat_template(
        _messages, tokenize=False, add_generation_prompt=True,
    )
    _inputs = tokenizer(_text, return_tensors="pt").to(model.device)
    try:
        _out = model.generate(
            **_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
        _new_tokens = _out[0][_inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(_new_tokens, skip_special_tokens=True).strip()
    except Exception as _e:
        print(f"  [MODEL ERROR] {_e}")
        return None


print("Prompt builders and LLM client ready.")

---
## Level 0 — raw source only
All cases.


In [ ]:
# CELL 9 — L0 generation: raw source only
L0_CASES = ALL_CASES

_l0_prog_file = L0_RESULTS_DIR / "l0_progress.json"
_l0_done = {}
if _l0_prog_file.exists():
    _l0_done = json.loads(_l0_prog_file.read_text())
    print(f"Resuming L0 — {len(_l0_done)} cases already done.")

print(f"\n{'='*60}")
print(f"L0 GENERATION: {len(L0_CASES)} cases | prompt=raw_code_only | model={MODEL_ID}")
print(f"{'='*60}\n")

for _i, _case in enumerate(L0_CASES, 1):
    _cid = _case["case_id"]
    if _cid in _l0_done:
        print(f"[{_i:3}/{len(L0_CASES)}] SKIP: {_cid}")
        continue

    print(f"[{_i:3}/{len(L0_CASES)}] {_cid}  CWE={_case['cwe_id']}")
    _prompt  = build_l0_prompt(_case["source"])
    _t0      = time.time()
    _raw     = call_model(_prompt)
    _elapsed = round(time.time() - _t0, 1)
    _fix     = extract_code(_raw) if _raw else ""

    _out_path = L0_OUTPUT_DIR / _case["vuln_dir"] / _case["filename"]
    _out_path.parent.mkdir(parents=True, exist_ok=True)
    _out_path.write_text(_fix)

    _l0_done[_cid] = {
        "case_id": _cid, "vuln_dir": _case["vuln_dir"], "filename": _case["filename"],
        "cwe_id": _case["cwe_id"], "status": "generated" if _raw else "error",
        "elapsed_s": _elapsed, "fix_len": len(_fix),
    }
    _l0_prog_file.write_text(json.dumps(_l0_done, indent=2))
    print(f"  -> {_l0_done[_cid]['status']} | {_elapsed}s | {len(_fix)} chars")

print(f"\nL0 generation complete: {len(_l0_done)}/{len(L0_CASES)}")

In [ ]:
# CELL 10 — L0 evaluation
_l0_results = []
_l0_counts  = {"CFR": 0, "PIR": 0, "IFR": 0}

print(f"Evaluating {len(L0_CASES)} L0 fixes...\n")

for _i, _case in enumerate(L0_CASES, 1):
    _out_path = L0_OUTPUT_DIR / _case["vuln_dir"] / _case["filename"]
    if not _out_path.exists():
        _cl = {"outcome": "IFR", "reason": "No output file", "orig_cwes": [], "fix_cwes": [], "new_cwes": []}
    else:
        _cl = classify_dvwa(_case["source"], _out_path.read_text(errors="replace"),
                            _case["filename"], _case["cwe_id"], _case["vuln_dir"])
    _l0_counts[_cl["outcome"]] += 1
    _l0_results.append({"case_id": _case["case_id"], "vuln_dir": _case["vuln_dir"],
                        "filename": _case["filename"], "cwe_id": _case["cwe_id"], **_cl})
    print(f"[{_i:3}/{len(L0_CASES)}] [{_cl['outcome']}] {_case['case_id']}  —  {_cl['reason']}")

_l0_total = len(_l0_results)
print(f"\n{'='*60}")
print(f"L0 Results — {_l0_total} cases")
print(f"  CFR: {_l0_counts['CFR']:3}  ({100*_l0_counts['CFR']/_l0_total:.1f}%)")
print(f"  PIR: {_l0_counts['PIR']:3}  ({100*_l0_counts['PIR']/_l0_total:.1f}%)")
print(f"  IFR: {_l0_counts['IFR']:3}  ({100*_l0_counts['IFR']/_l0_total:.1f}%)")
_print_level_table(_l0_results)

_l0_final = {
    "level": "L0", "model": MODEL_ID, "prompt_type": "raw_code_only",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "total_cases": _l0_total,
    "summary": {"CFR": _l0_counts["CFR"], "PIR": _l0_counts["PIR"], "IFR": _l0_counts["IFR"],
                "CFR_pct": round(100*_l0_counts["CFR"]/_l0_total, 1),
                "PIR_pct": round(100*_l0_counts["PIR"]/_l0_total, 1),
                "IFR_pct": round(100*_l0_counts["IFR"]/_l0_total, 1)},
    "cases": _l0_results,
}
(L0_RESULTS_DIR / "l0_results.json").write_text(json.dumps(_l0_final, indent=2))
print(f"\nL0 results saved -> {L0_RESULTS_DIR / 'l0_results.json'}")

---
## Level 1 — source + SAST + application context
Non-CFR cases from L0.


In [ ]:
# CELL 11 — Build L1 case list
_l0_res_file = L0_RESULTS_DIR / "l0_results.json"
if not _l0_res_file.exists():
    raise RuntimeError("L0 results not found — run CELL 10 first.")

_l0_data        = json.loads(_l0_res_file.read_text())
_l0_non_cfr_ids = {c["case_id"] for c in _l0_data["cases"] if c["outcome"] != "CFR"}
L1_CASES        = [c for c in ALL_CASES if c["case_id"] in _l0_non_cfr_ids]

print(f"L0 total: {_l0_data['total_cases']} | CFR: {_l0_data['summary']['CFR']} | escalated to L1: {len(L1_CASES)}")
print(f"\nL1 cases ({len(L1_CASES)}):")
for _c in L1_CASES:
    _prev = next(r for r in _l0_data["cases"] if r["case_id"] == _c["case_id"])
    _ctx  = "context" if _c["frontend_context"] else ""
    print(f"  {_c['case_id']:<35}  L0={_prev['outcome']}  sast={len(_c['sast_findings'])}  {_ctx}")

In [ ]:
# CELL 12 — L1 generation: source + SAST + application context
_l1_prog_file = L1_RESULTS_DIR / "l1_progress.json"
_l1_done = {}
if _l1_prog_file.exists():
    _l1_done = json.loads(_l1_prog_file.read_text())
    print(f"Resuming L1 — {len(_l1_done)} cases already done.")

print(f"\n{'='*60}")
print(f"L1 GENERATION: {len(L1_CASES)} cases | prompt=code+sast+context | model={MODEL_ID}")
print(f"{'='*60}\n")

for _i, _case in enumerate(L1_CASES, 1):
    _cid = _case["case_id"]
    if _cid in _l1_done:
        print(f"[{_i:3}/{len(L1_CASES)}] SKIP: {_cid}")
        continue

    _has_ctx = bool(_case["frontend_context"])
    print(f"[{_i:3}/{len(L1_CASES)}] {_cid}  CWE={_case['cwe_id']}  sast={len(_case['sast_findings'])}  ctx={'yes' if _has_ctx else 'no'}")
    _sast_str = format_sast_for_prompt(_case["sast_findings"])
    _prompt   = build_l1_prompt(_case["source"], _sast_str, _case["frontend_context"])
    _t0       = time.time()
    _raw      = call_model(_prompt)
    _elapsed  = round(time.time() - _t0, 1)
    _fix      = extract_code(_raw) if _raw else ""

    _out_path = L1_OUTPUT_DIR / _case["vuln_dir"] / _case["filename"]
    _out_path.parent.mkdir(parents=True, exist_ok=True)
    _out_path.write_text(_fix)

    _l1_done[_cid] = {
        "case_id": _cid, "vuln_dir": _case["vuln_dir"], "filename": _case["filename"],
        "cwe_id": _case["cwe_id"], "sast_findings": len(_case["sast_findings"]),
        "has_context": _has_ctx, "status": "generated" if _raw else "error",
        "elapsed_s": _elapsed, "fix_len": len(_fix),
    }
    _l1_prog_file.write_text(json.dumps(_l1_done, indent=2))
    print(f"  -> {_l1_done[_cid]['status']} | {_elapsed}s | {len(_fix)} chars")

print(f"\nL1 generation complete: {len(_l1_done)}/{len(L1_CASES)}")

In [ ]:
# CELL 13 — L1 evaluation
_l1_results = []
_l1_counts  = {"CFR": 0, "PIR": 0, "IFR": 0}

print(f"Evaluating {len(L1_CASES)} L1 fixes...\n")

for _i, _case in enumerate(L1_CASES, 1):
    _out_path = L1_OUTPUT_DIR / _case["vuln_dir"] / _case["filename"]
    if not _out_path.exists():
        _cl = {"outcome": "IFR", "reason": "No output file", "orig_cwes": [], "fix_cwes": [], "new_cwes": []}
    else:
        _cl = classify_dvwa(_case["source"], _out_path.read_text(errors="replace"),
                            _case["filename"], _case["cwe_id"], _case["vuln_dir"])
    _l1_counts[_cl["outcome"]] += 1
    _l1_results.append({"case_id": _case["case_id"], "vuln_dir": _case["vuln_dir"],
                        "filename": _case["filename"], "cwe_id": _case["cwe_id"], **_cl})
    print(f"[{_i:3}/{len(L1_CASES)}] [{_cl['outcome']}] {_case['case_id']}  —  {_cl['reason']}")

_l1_total = max(len(_l1_results), 1)
print(f"\n{'='*60}")
print(f"L1 Results — {len(_l1_results)} cases (escalated from L0)")
if _l1_results:
    print(f"  CFR: {_l1_counts['CFR']:3}  ({100*_l1_counts['CFR']/_l1_total:.1f}%)")
    print(f"  PIR: {_l1_counts['PIR']:3}  ({100*_l1_counts['PIR']/_l1_total:.1f}%)")
    print(f"  IFR: {_l1_counts['IFR']:3}  ({100*_l1_counts['IFR']/_l1_total:.1f}%)")
_print_level_table(_l1_results)

_l1_final = {
    "level": "L1", "model": MODEL_ID, "prompt_type": "raw_code_plus_sast_plus_context",
    "timestamp": datetime.now(timezone.utc).isoformat(), "total_cases": len(_l1_results),
    "summary": {"CFR": _l1_counts["CFR"], "PIR": _l1_counts["PIR"], "IFR": _l1_counts["IFR"],
                "CFR_pct": round(100*_l1_counts["CFR"]/_l1_total, 1),
                "PIR_pct": round(100*_l1_counts["PIR"]/_l1_total, 1),
                "IFR_pct": round(100*_l1_counts["IFR"]/_l1_total, 1)},
    "cases": _l1_results,
}
(L1_RESULTS_DIR / "l1_results.json").write_text(json.dumps(_l1_final, indent=2))
print(f"\nL1 results saved -> {L1_RESULTS_DIR / 'l1_results.json'}")

---
## Level 2 — source + SAST + DAST + application context + repair guidance
Non-CFR cases from L1.


In [ ]:
# CELL 14 — Build L2 case list
_l1_res_file = L1_RESULTS_DIR / "l1_results.json"
if not _l1_res_file.exists():
    raise RuntimeError("L1 results not found — run CELL 13 first.")

_l1_data        = json.loads(_l1_res_file.read_text())
_l1_non_cfr_ids = {c["case_id"] for c in _l1_data["cases"] if c["outcome"] != "CFR"}
L2_CASES        = [c for c in ALL_CASES if c["case_id"] in _l1_non_cfr_ids]

print(f"L1 total: {_l1_data['total_cases']} | CFR: {_l1_data['summary']['CFR']} | escalated to L2: {len(L2_CASES)}")
print(f"\nL2 cases ({len(L2_CASES)}):")
for _c in L2_CASES:
    _prev     = next(r for r in _l1_data["cases"] if r["case_id"] == _c["case_id"])
    _hint_key = f"{_c['vuln_dir']}_{Path(_c['filename']).stem}"
    _hint     = "hint" if (_hint_key in VULN_L2_HINTS or _c["vuln_dir"] in VULN_L2_HINTS) else ""
    _ctx      = "context" if _c["frontend_context"] else ""
    print(f"  {_c['case_id']:<35}  L1={_prev['outcome']}  zap={len(_c['dast_zap'])}  {_ctx}  {_hint}  [hint_key={_hint_key}]")

In [ ]:
# CELL 15 — L2 generation: source + SAST + DAST + application context + repair guidance
_l2_prog_file = L2_RESULTS_DIR / "l2_progress.json"
_l2_done = {}
if _l2_prog_file.exists():
    _l2_done = json.loads(_l2_prog_file.read_text())
    print(f"Resuming L2 — {len(_l2_done)} cases already done.")

print(f"\n{'='*60}")
print(f"L2 GENERATION: {len(L2_CASES)} cases | prompt=code+sast+dast+context+hint | model={MODEL_ID}")
print(f"{'='*60}\n")

for _i, _case in enumerate(L2_CASES, 1):
    _cid = _case["case_id"]
    if _cid in _l2_done:
        print(f"[{_i:3}/{len(L2_CASES)}] SKIP: {_cid}")
        continue

    _hint_key = f"{_case['vuln_dir']}_{Path(_case['filename']).stem}"
    _hint = VULN_L2_HINTS.get(_hint_key, VULN_L2_HINTS.get(_case["vuln_dir"], ""))
    print(f"[{_i:3}/{len(L2_CASES)}] {_cid}  zap={len(_case['dast_zap'])}  hint={'yes' if _hint else 'no'}")
    _sast_str = format_sast_for_prompt(_case["sast_findings"])
    _dast_str = format_dast_for_prompt(_case["dast_zap"], _case["dast_ffuf"])
    _prompt   = build_l2_prompt(_case["source"], _sast_str, _dast_str,
                                 _case["frontend_context"], _hint)
    _t0       = time.time()
    _raw      = call_model(_prompt)
    _elapsed  = round(time.time() - _t0, 1)
    _fix      = extract_code(_raw) if _raw else ""

    _out_path = L2_OUTPUT_DIR / _case["vuln_dir"] / _case["filename"]
    _out_path.parent.mkdir(parents=True, exist_ok=True)
    _out_path.write_text(_fix)

    _l2_done[_cid] = {
        "case_id": _cid, "vuln_dir": _case["vuln_dir"], "filename": _case["filename"],
        "cwe_id": _case["cwe_id"], "sast_findings": len(_case["sast_findings"]),
        "dast_zap": len(_case["dast_zap"]), "has_hint": bool(_hint),
        "status": "generated" if _raw else "error", "elapsed_s": _elapsed, "fix_len": len(_fix),
    }
    _l2_prog_file.write_text(json.dumps(_l2_done, indent=2))
    print(f"  -> {_l2_done[_cid]['status']} | {_elapsed}s | {len(_fix)} chars")

print(f"\nL2 generation complete: {len(_l2_done)}/{len(L2_CASES)}")

In [ ]:
# CELL 16 — L2 evaluation
_l2_results = []
_l2_counts  = {"CFR": 0, "PIR": 0, "IFR": 0}

print(f"Evaluating {len(L2_CASES)} L2 fixes...\n")

for _i, _case in enumerate(L2_CASES, 1):
    _out_path = L2_OUTPUT_DIR / _case["vuln_dir"] / _case["filename"]
    if not _out_path.exists():
        _cl = {"outcome": "IFR", "reason": "No output file", "orig_cwes": [], "fix_cwes": [], "new_cwes": []}
    else:
        _cl = classify_dvwa(_case["source"], _out_path.read_text(errors="replace"),
                            _case["filename"], _case["cwe_id"], _case["vuln_dir"])
    _l2_counts[_cl["outcome"]] += 1
    _l2_results.append({"case_id": _case["case_id"], "vuln_dir": _case["vuln_dir"],
                        "filename": _case["filename"], "cwe_id": _case["cwe_id"], **_cl})
    print(f"[{_i:3}/{len(L2_CASES)}] [{_cl['outcome']}] {_case['case_id']}  —  {_cl['reason']}")

_l2_total = max(len(_l2_results), 1)
print(f"\n{'='*60}")
print(f"L2 Results — {len(_l2_results)} cases (escalated from L1)")
if _l2_results:
    print(f"  CFR: {_l2_counts['CFR']:3}  ({100*_l2_counts['CFR']/_l2_total:.1f}%)")
    print(f"  PIR: {_l2_counts['PIR']:3}  ({100*_l2_counts['PIR']/_l2_total:.1f}%)")
    print(f"  IFR: {_l2_counts['IFR']:3}  ({100*_l2_counts['IFR']/_l2_total:.1f}%)")
_print_level_table(_l2_results)

_l2_final = {
    "level": "L2", "model": MODEL_ID, "prompt_type": "raw_code_plus_sast_plus_dast_plus_hint",
    "timestamp": datetime.now(timezone.utc).isoformat(), "total_cases": len(_l2_results),
    "summary": {"CFR": _l2_counts["CFR"], "PIR": _l2_counts["PIR"], "IFR": _l2_counts["IFR"],
                "CFR_pct": round(100*_l2_counts["CFR"]/_l2_total, 1),
                "PIR_pct": round(100*_l2_counts["PIR"]/_l2_total, 1),
                "IFR_pct": round(100*_l2_counts["IFR"]/_l2_total, 1)},
    "cases": _l2_results,
}
(L2_RESULTS_DIR / "l2_results.json").write_text(json.dumps(_l2_final, indent=2))
print(f"\nL2 results saved -> {L2_RESULTS_DIR / 'l2_results.json'}")

---
## Cross-level summary


In [ ]:
# CELL 17 — Cross-level summary
print("=" * 70)
print(f"DVWA LLM PIPELINE v3 — FINAL SUMMARY")
print(f"Model: {MODEL_ID}")
print("=" * 70)

_all_level_data = {}
for _level, _rdir in [("L0", L0_RESULTS_DIR), ("L1", L1_RESULTS_DIR), ("L2", L2_RESULTS_DIR)]:
    _rf = _rdir / f"{_level.lower()}_results.json"
    if _rf.exists():
        _all_level_data[_level] = json.loads(_rf.read_text())

print(f"\n  {'Level':<6} {'Cases':>6} {'CFR':>5} {'CFR%':>6} {'PIR':>5} {'PIR%':>6} {'IFR':>5} {'IFR%':>6}")
print("  " + "-" * 50)
for _level in ["L0", "L1", "L2"]:
    if _level not in _all_level_data:
        print(f"  {_level}      —  (not run yet)")
        continue
    _d, _s, _n = _all_level_data[_level], _all_level_data[_level]["summary"], _all_level_data[_level]["total_cases"]
    print(f"  {_level}    {_n:>6} {_s['CFR']:>5} {_s['CFR_pct']:>5.1f}% "
          f"{_s['PIR']:>5} {_s['PIR_pct']:>5.1f}% {_s['IFR']:>5} {_s['IFR_pct']:>5.1f}%")

_cfr_l0 = {c["case_id"] for c in _all_level_data.get("L0",{}).get("cases",[]) if c["outcome"]=="CFR"}
_cfr_l1 = {c["case_id"] for c in _all_level_data.get("L1",{}).get("cases",[]) if c["outcome"]=="CFR"}
_cfr_l2 = {c["case_id"] for c in _all_level_data.get("L2",{}).get("cases",[]) if c["outcome"]=="CFR"}
_cumulative  = _cfr_l0 | _cfr_l1 | _cfr_l2
_total_cases = len(ALL_CASES)

print(f"\n── Cumulative across all levels ──")
print(f"  Fixed at L0:                  {len(_cfr_l0):3}  ({100*len(_cfr_l0)/_total_cases:.1f}%)")
print(f"  Additionally fixed at L1:     {len(_cfr_l1):3}  ({100*len(_cfr_l1)/_total_cases:.1f}%)")
print(f"  Additionally fixed at L2:     {len(_cfr_l2):3}  ({100*len(_cfr_l2)/_total_cases:.1f}%)")
print(f"  Total fixed (any level):      {len(_cumulative):3}  ({100*len(_cumulative)/_total_cases:.1f}%)")
print(f"  Never fixed:                  {_total_cases - len(_cumulative):3}  ({100*(_total_cases-len(_cumulative))/_total_cases:.1f}%)")

print(f"\n── Per-vulnerability-folder breakdown ──")
print(f"  {'Vuln folder':<25} {'Total':>6} {'L0_CFR':>7} {'L1_CFR':>7} {'L2_CFR':>7} {'Any_CFR':>8}")
print("  " + "-" * 60)
for _vd in sorted(set(c["vuln_dir"] for c in ALL_CASES)):
    _vd_ids = {c["case_id"] for c in ALL_CASES if c["vuln_dir"] == _vd}
    _n   = len(_vd_ids)
    _l0c = len(_cfr_l0 & _vd_ids)
    _l1c = len(_cfr_l1 & _vd_ids)
    _l2c = len(_cfr_l2 & _vd_ids)
    _any = len((_cfr_l0 | _cfr_l1 | _cfr_l2) & _vd_ids)
    print(f"  {_vd:<25} {_n:>6} {_l0c:>7} {_l1c:>7} {_l2c:>7} {_any:>8}")

---
## Post-hoc validation and ablation (CELLS 18-23)

CELLS 18, 19, 22 and 23 are analysis only and need no GPU; they run on the already-generated outputs.
CELL 20 generates 6 new completions for the Level 2 ablation.


In [ ]:
# CELL 18 — Post-hoc validation verdicts
# Four independent checks applied to the saved patches after the cascade had run.
# Each verdict is recorded here so the correction is reproducible and auditable.

# --- 1. manual review of the cases the automated harness could not verify ---
MANUAL_REVIEW = {
    "sqli_high":    ("CFR",      "prepared statement with bound parameter"),
    "sqli_medium":  ("CFR",      "prepared statements, MySQL and SQLite branches"),
    "csp_medium":   ("CFR",      "removed 'unsafe-inline' and nonce; header matches impossible.php"),
    "api_low":      ("PIR",      "FILTER_SANITIZE_URL preserves quotes; target weakness not in this file"),
    "api_medium":   ("PIR",      "closes an incidental PHP_SELF XSS; API privilege escalation untouched"),
    "bac_high":     ("PIR",      "audit-log change only; no session_regenerate_id for session fixation"),
    "brute_high":   ("PIR",      "output escaping only; no failed_login counter or lockout"),
    "brute_medium": ("PIR",      "prepared statements only; no lockout; FILTER_SANITIZE_STRING deprecated"),
    "csp_high":     ("PIR",      "CSP already script-src 'self'; jsonp.php callback path untouched"),
    "csp_low":      ("PIR",      "FILTER_VALIDATE_URL checks syntax only; external script still included"),
    "api_high":     ("EXCLUDED", "static informational page; no user input, nothing to repair"),
}

# --- 2. exploit replay against the running application ----------------------
# MITIGATED = the payload no longer succeeds; STILL_EXPLOITABLE = it still does.
EXPLOIT_REPLAY = {}                                   # L0 patches, 7 modules x 3 levels
for _m in ("sqli", "sqli_blind", "exec", "xss_r", "fi"):
    for _l in ("low", "medium", "high"):
        EXPLOIT_REPLAY[f"{_m}_{_l}"] = "MITIGATED"
EXPLOIT_REPLAY.update({
    "xss_s_low": "MITIGATED", "xss_s_medium": "MITIGATED",
    "xss_s_high": "STILL_EXPLOITABLE",            # <img onerror> passes the name filter
    "open_redirect_low":    "STILL_EXPLOITABLE",  # FILTER_VALIDATE_URL accepts http://evil.com
    "open_redirect_medium": "STILL_EXPLOITABLE",  # //evil.com bypasses the scheme block
    "open_redirect_high":   "STILL_EXPLOITABLE",  # substring check on "info.php"
})

EXPLOIT_REPLAY_L2 = {                                 # L2 patches, deterministic payloads
    "open_redirect_low":    "MITIGATED",              # run 2026-09-07; all baselines
    "open_redirect_medium": "MITIGATED",              # confirmed exploitable first.
    "open_redirect_high":   "MITIGATED",              # See l2_exploit_replay.json
    "fi_high":              "MITIGATED",
}

# --- 3. functional regression: patches that introduced a new PHP error ------
SMOKE_BROKEN = {
    ("l0", "authbypass_high"):   "redundant session_start()",
    ("l0", "authbypass_medium"): "redundant session_start()",
    ("l0", "brute_medium"):      "FILTER_SANITIZE_STRING deprecated in PHP 8.1+",
    ("l0", "weak_id_high"):      "redundant session_start()",
    ("l0", "weak_id_low"):       "redundant session_start()",
    ("l1", "weak_id_low"):       "redundant session_start()",
    ("l2", "weak_id_low"):       "redundant session_start()",
}

# the baseline already errored, so no verdict can be attributed to the patch
SMOKE_INCONCLUSIVE = {("l0", "brute_high"), ("l0", "captcha_low"),
                      ("l0", "captcha_medium"), ("l0", "captcha_high")}

# --- 4. target-weakness check: recorded target CWE != the module's weakness --
# See Section 6.5.1. bac's target was mis-specified in VULN_CWE_MAP; brute_low's
# was substituted from its first SAST finding by the assignment in CELL 6.
TARGET_WEAKNESS_MISMATCH = {
    "bac_low":    ("PIR", "assessed against CWE-89; cookie-based authorisation unchanged (CWE-639)"),
    "bac_medium": ("PIR", "assessed against CWE-89; static token check unchanged (CWE-639)"),
    "brute_low":  ("PIR", "assessed against CWE-89; no rate limiting introduced (CWE-307)"),
}

print(f"manual review          : {len(MANUAL_REVIEW)} cases")
print(f"exploit replay         : {len(EXPLOIT_REPLAY)} L0, {len(EXPLOIT_REPLAY_L2)} L2")
print(f"regression broken      : {len(SMOKE_BROKEN)} patches")
print(f"regression inconclusive: {len(SMOKE_INCONCLUSIVE)} patches")
print(f"target-weakness        : {len(TARGET_WEAKNESS_MISMATCH)} cases")


In [ ]:
# CELL 19 — Apply the validation and rebuild the corrected tables
import json
from pathlib import Path
from collections import defaultdict

_l0 = json.loads((L0_RESULTS_DIR / "l0_results.json").read_text())
_l1 = json.loads((L1_RESULTS_DIR / "l1_results.json").read_text())
_l2 = json.loads((L2_RESULTS_DIR / "l2_results.json").read_text())

def _key(rec):
    """case_id -> module_level key used by the validation dictionaries."""
    return f"{rec['vuln_dir']}_{Path(rec['filename']).stem}"

def _apply(records, layer):
    out, changes = [], []
    for r in records:
        r = dict(r); k = _key(r); before = r["outcome"]

        if layer == "l0" and k in MANUAL_REVIEW:
            v, why = MANUAL_REVIEW[k]
            r["outcome"], r["oracle"], r["reason"] = v, "manual_review", why

        if layer == "l0" and EXPLOIT_REPLAY.get(k) == "STILL_EXPLOITABLE" \
           and r["outcome"] == "CFR":
            r["outcome"], r["oracle"] = "PIR", "exploit_replay"
            r["reason"] = "payload still succeeds against the patched file"

        if layer == "l0" and k in TARGET_WEAKNESS_MISMATCH and r["outcome"] == "CFR":
            v, why = TARGET_WEAKNESS_MISMATCH[k]
            r["outcome"], r["oracle"], r["reason"] = v, "target_weakness_check", why

        if (layer, k) in SMOKE_BROKEN and r["outcome"] == "CFR":
            r["outcome"], r["oracle"] = "PIR", "functional_smoke_test"
            r["reason"] = SMOKE_BROKEN[(layer, k)]

        if r["outcome"] != before:
            changes.append((k, before, r["outcome"], r.get("oracle"), r["reason"]))
        out.append(r)
    return out, changes

_l0c, _ch0 = _apply(_l0["cases"], "l0")
_l1c, _ch1 = _apply(_l1["cases"], "l1")
_l2c, _ch2 = _apply(_l2["cases"], "l2")

print("RECLASSIFICATIONS\n")
for lay, ch in (("L0", _ch0), ("L1", _ch1), ("L2", _ch2)):
    for k, b, a, o, why in ch:
        print(f"  {lay}  {k:22s} {b} -> {a:8s} [{o}]  {why}")

def _tally(recs):
    t = defaultdict(int)
    for r in recs: t[r["outcome"]] += 1
    return dict(t)

_excluded = [r for r in _l0c if r["outcome"] == "EXCLUDED"]
_l0_eval  = [r for r in _l0c if r["outcome"] != "EXCLUDED"]

print(f"\nexcluded from the dataset: {[_key(r) for r in _excluded]}")
print(f"\nL0 corrected: N={len(_l0_eval)}  {_tally(_l0_eval)}")
print(f"L1 as run   : N={len(_l1c)}  {_tally(_l1c)}")
print(f"L2 corrected: N={len(_l2c)}  {_tally(_l2c)}")

_c0 = _tally(_l0_eval).get("CFR", 0)
_c1 = _tally(_l1c).get("CFR", 0)
_c2 = _tally(_l2c).get("CFR", 0)
_N  = len(_l0_eval)
print(f"\ncumulative: L0 {_c0}/{_N} = {_c0/_N*100:.1f}%"
      f" | +L1 {_c0+_c1}/{_N} = {(_c0+_c1)/_N*100:.1f}%"
      f" | +L2 {_c0+_c1+_c2}/{_N} = {(_c0+_c1+_c2)/_N*100:.1f}%")

_not_escalated = [_key(r) for r in _l0_eval
                  if r["outcome"] != "CFR"
                  and _key(r) not in {_key(x) for x in _l1c}]
print(f"\nreclassified after the run, therefore never escalated ({len(_not_escalated)}):")
print(f"  {_not_escalated}")
print("  -> the corrected cumulative rate is a LOWER BOUND; state this in the paper")

# per-module table for the corrected L0
print("\nCORRECTED L0 PER-MODULE")
_by = defaultdict(lambda: defaultdict(int))
for r in _l0_eval: _by[r["vuln_dir"]][r["outcome"]] += 1
for m in sorted(_by):
    d = _by[m]; n = sum(d.values())
    print(f"  {m:15s} N={n}  CFR={d.get('CFR',0)}  PIR={d.get('PIR',0)}  IFR={d.get('IFR',0)}")

(L0_RESULTS_DIR / "l0_results_validated.json").write_text(json.dumps(
    {**_l0, "cases": _l0c, "validation": "manual_review + exploit_replay + smoke_test"}, indent=2))
(L2_RESULTS_DIR / "l2_results_validated.json").write_text(json.dumps(
    {**_l2, "cases": _l2c, "validation": "smoke_test"}, indent=2))
print("\nsaved l0_results_validated.json and l2_results_validated.json")

In [ ]:
# CELL 20 — Ablation generation: separates DAST evidence from repair guidance
# The three open_redirect cases are the only ones carrying both a ZAP finding and
# repair guidance, so they are the only cases where all four conditions exist.
#   arm nodast: guidance kept, DAST removed
#   arm nohint: DAST kept, guidance removed
# 6 completions. The original L2 outputs are not touched.

ABLATION_CASES = ["open_redirect_low", "open_redirect_medium", "open_redirect_high"]

ABLATE_DIRS = {
    "nodast": L2_OUTPUT_DIR.parent / (L2_OUTPUT_DIR.name + "_ablate_nodast"),
    "nohint": L2_OUTPUT_DIR.parent / (L2_OUTPUT_DIR.name + "_ablate_nohint"),
}
for _d in ABLATE_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

_abl_cases = [c for c in L2_CASES
              if f"{c['vuln_dir']}_{Path(c['filename']).stem}" in ABLATION_CASES]
print(f"ablation cases: {[c['case_id'] for c in _abl_cases]}\n")
assert len(_abl_cases) == 3, "expected the three open_redirect cases"

_abl_prog = {}
for _arm in ("nodast", "nohint"):
    print(f"{'='*60}\nARM {_arm.upper()}  |  model={MODEL_ID}\n{'='*60}")
    for _case in _abl_cases:
        _k        = f"{_case['vuln_dir']}_{Path(_case['filename']).stem}"
        _hint     = VULN_L2_HINTS.get(_k, VULN_L2_HINTS.get(_case["vuln_dir"], ""))
        _sast_str = format_sast_for_prompt(_case["sast_findings"])
        _dast_str = format_dast_for_prompt(_case["dast_zap"], _case["dast_ffuf"])

        if _arm == "nodast":
            _dast_str = "No DAST findings."      # only this field changes
        else:
            _hint = ""                            # only this field changes

        _prompt = build_l2_prompt(_case["source"], _sast_str, _dast_str,
                                  _case["frontend_context"], _hint)
        _t0  = time.time()
        _raw = call_model(_prompt)
        _fix = extract_code(_raw) if _raw else ""
        _p   = ABLATE_DIRS[_arm] / _case["vuln_dir"] / _case["filename"]
        _p.parent.mkdir(parents=True, exist_ok=True)
        _p.write_text(_fix)
        _abl_prog[(_arm, _k)] = {"elapsed_s": round(time.time()-_t0, 1), "fix_len": len(_fix)}
        print(f"  {_k:24s} -> {len(_fix):5d} chars | {_abl_prog[(_arm,_k)]['elapsed_s']}s")

print(f"\nablation generation complete: {len(_abl_prog)} completions")

In [ ]:
# CELL 21 — Ablation evaluation and the 2x2
_abl_out = {}
for _arm in ("nodast", "nohint"):
    for _case in _abl_cases:
        _k = f"{_case['vuln_dir']}_{Path(_case['filename']).stem}"
        _p = ABLATE_DIRS[_arm] / _case["vuln_dir"] / _case["filename"]
        if not _p.exists():
            _cl = {"outcome": "IFR", "reason": "no output file"}
        else:
            _cl = classify_dvwa(_case["source"], _p.read_text(errors="replace"),
                                _case["filename"], _case["cwe_id"], _case["vuln_dir"])
        _abl_out[(_arm, _k)] = _cl["outcome"]

_l1_by = {f"{c['vuln_dir']}_{Path(c['filename']).stem}": c["outcome"] for c in _l1c}
_l2_by = {f"{c['vuln_dir']}_{Path(c['filename']).stem}": c["outcome"] for c in _l2c}

print("2x2 ABLATION ON THE THREE open_redirect CASES\n")
print(f"  {'case':24s}{'A: neither':>13}{'B: hint only':>15}{'C: DAST only':>15}{'D: both':>10}")
for _k in ABLATION_CASES:
    print(f"  {_k:24s}{_l1_by.get(_k,'-'):>13}{_abl_out.get(('nodast',_k),'-'):>15}"
          f"{_abl_out.get(('nohint',_k),'-'):>15}{_l2_by.get(_k,'-'):>10}")

_cnt = lambda d: sum(1 for v in d if v == "CFR")
print(f"\n  A (neither, = L1)  : {_cnt([_l1_by.get(k) for k in ABLATION_CASES])}/3")
print(f"  B (hint, no DAST)  : {_cnt([_abl_out.get(('nodast',k)) for k in ABLATION_CASES])}/3")
print(f"  C (DAST, no hint)  : {_cnt([_abl_out.get(('nohint',k)) for k in ABLATION_CASES])}/3")
print(f"  D (both,   = L2)   : {_cnt([_l2_by.get(k) for k in ABLATION_CASES])}/3")

json.dump({f"{a}|{k}": v for (a, k), v in _abl_out.items()},
          open(L2_RESULTS_DIR / "ablation_results.json", "w"), indent=2)
print(f"\nsaved -> {L2_RESULTS_DIR / 'ablation_results.json'}")

In [ ]:
# CELL 22 — L2 attribution cross-tabulation
# Classifies every L2 case by what was ACTUALLY present in its prompt.
_rows = []
for _case in L2_CASES:
    _k    = f"{_case['vuln_dir']}_{Path(_case['filename']).stem}"
    _hint = bool(VULN_L2_HINTS.get(_k, VULN_L2_HINTS.get(_case["vuln_dir"], "")))
    _dast = len(_case["dast_zap"]) > 0
    _rows.append({"case": _k, "dast": _dast, "hint": _hint,
                  "outcome": _l2_by.get(_k, "?")})

print(f"  {'case':24s}{'ZAP':>6}{'hint':>7}{'L2':>7}")
for _r in sorted(_rows, key=lambda r: (not r["dast"], not r["hint"], r["case"])):
    print(f"  {_r['case']:24s}{str(_r['dast']):>6}{str(_r['hint']):>7}{_r['outcome']:>7}")

def _cell(dast, hint):
    g = [r for r in _rows if r["dast"] == dast and r["hint"] == hint]
    return sum(1 for r in g if r["outcome"] == "CFR"), len(g)

print("\nATTRIBUTION (observational: cells contain different modules)")
print(f"  A  neither              : 0/{len(_rows)} by construction (all failed at L1)")
print("  B  hint, no ZAP evidence: {}/{}".format(*_cell(False, True)))
print("  C  ZAP evidence, no hint: {}/{}".format(*_cell(True, False)))
print("  D  both                 : {}/{}".format(*_cell(True, True)))

In [ ]:
# CELL 23 — Wilson confidence intervals + final export
from math import sqrt

def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0.0, c-h)*100, min(1.0, c+h)*100)

_n0, _n1, _n2 = len(_l0_eval), len(_l1c), len(_l2c)
_rates = [
    ("L0 fix rate",         _c0, _n0),
    ("L1 recovery rate",    _c1, _n1),
    ("L2 recovery rate",    _c2, _n2),
    ("Cumulative after L0", _c0,           _n0),
    ("Cumulative after L1", _c0+_c1,       _n0),
    ("Cumulative after L2", _c0+_c1+_c2,   _n0),
]
print(f"  {'rate':24s}{'k/n':>10}{'value':>9}{'Wilson 95% CI':>22}")
for _lab, _k, _n in _rates:
    _lo, _hi = wilson(_k, _n)
    print(f"  {_lab:24s}{f'{_k}/{_n}':>10}{_k/_n*100:>8.1f}%{f'[{_lo:.1f}, {_hi:.1f}]':>22}")

print("\nNOTE: the three security-level variants of a module are not independent;")
print("at the effective sample size of 18 modules these intervals are wider still.")

_summary = {
    "dataset": {"files_evaluated": _n0, "excluded": [_key(r) for r in _excluded],
                "modules": len({r["vuln_dir"] for r in _l0_eval})},
    "L0": _tally(_l0_eval), "L1": _tally(_l1c), "L2": _tally(_l2c),
    "cumulative": {"after_L0": _c0, "after_L1": _c0+_c1, "after_L2": _c0+_c1+_c2,
                   "denominator": _n0},
    "not_escalated_after_reclassification": _not_escalated,
    "validation": {"manual_review": len(MANUAL_REVIEW),
                   "exploit_replay": len(EXPLOIT_REPLAY),
                   "smoke_test_broken": len(SMOKE_BROKEN)},
}
(L2_RESULTS_DIR / "final_summary.json").write_text(json.dumps(_summary, indent=2, default=str))
print(f"\nsaved -> {L2_RESULTS_DIR / 'final_summary.json'}")

---
## Oracle provenance (CELL 24)

Records which oracle decided every classification, so a correct fix can be reported
with its evidential basis. No GPU and no re-scanning: `classify_dvwa()` already
stamped the deciding branch into each `reason` string.
Requires CELL 2, CELL 18 and CELL 19 to have been run.


In [ ]:
# CELL 24 — Oracle provenance for every classification

import csv, json, re
from pathlib import Path
from collections import defaultdict

_raw_files = {
    "l0": L0_RESULTS_DIR / "l0_results.json",
    "l1": L1_RESULTS_DIR / "l1_results.json",
    "l2": L2_RESULTS_DIR / "l2_results.json",
}
_orig_reason = {}
for _lay, _p in _raw_files.items():
    for _r in json.loads(_p.read_text())["cases"]:
        _orig_reason[(_lay, _key(_r))] = _r.get("reason", "")


def _tag_of(reason):
    _m = re.search(r"\[([^\]]+)\]", reason or "")
    return _m.group(1) if _m else ""

def _precheck_reason(raw):
    _t = (raw or "").strip()
    if _t.startswith("PHP syntax error"):
        return "PHP syntax error (php -l), rejected before harness"
    _map = {
        "Empty output":
            "empty output, rejected by pre-check before harness",
        "Identical to original":
            "duplicate output (unmodified copy of source), rejected by pre-check before harness",
        "Code body deleted (< 30% original size)":
            "response below 30% length threshold, rejected by pre-check before harness",
    }
    return _map.get(_t, _t or "invalid output, rejected before harness")
# ────────────────────────────────────────────────────────────────────────────


def _prompt_context(case_key, vuln_dir, layer):
    """Which repair-property blocks were in this case's prompt.

    Application context enters at L1 and L2; repair guidance at L2 only.
    """
    _fc = VULN_FRONTEND_CONTEXT.get(case_key, VULN_FRONTEND_CONTEXT.get(vuln_dir, ""))
    _hint = VULN_L2_HINTS.get(case_key, VULN_L2_HINTS.get(vuln_dir, ""))
    if layer == "l0":
        return False, False
    if layer == "l1":
        return bool(_fc), False
    return bool(_fc), bool(_hint)


def _smoke_status(case_key, layer):
    if (layer, case_key) in SMOKE_BROKEN:
        return "broken"
    if (layer, case_key) in SMOKE_INCONCLUSIVE:
        return "inconclusive"
    return "passed"

def _provenance(rec, layer):
    _k = _key(rec)
    _tag = _tag_of(_orig_reason.get((layer, _k), ""))
    _fc, _hint = _prompt_context(_k, rec["vuln_dir"], layer)

    if rec["outcome"] == "IFR":
        return "pre_check", False, "IFR: " + _precheck_reason(
            _orig_reason.get((layer, _k), "") or rec.get("reason", ""))
    # ────────────────────────────────────────────────────────────────────────

    if layer == "l0" and _k in EXPLOIT_REPLAY:
        return "replay", True, f"exploit replay (L0): {EXPLOIT_REPLAY[_k]}"
    if layer == "l2" and _k in EXPLOIT_REPLAY_L2:
        return "replay", True, f"exploit replay (L2): {EXPLOIT_REPLAY_L2[_k]}"
    if layer == "l0" and _k in TARGET_WEAKNESS_MISMATCH:
        return ("target_weakness_mismatch", True,
                "recorded target CWE is not the module's weakness")
    if layer == "l0" and _k in MANUAL_REVIEW:
        return "manual_review", True, "independent manual review vs DVWA reference"
    if _tag == "semgrep":
        return "scanner", True, "target CWE present in original, absent after patch"
    if _tag.startswith("pattern:"):
        _blocks = [_b for _b, _on in (("frontend context", _fc), ("fix hint", _hint)) if _on]
        if _blocks:
            return ("verifier-conformity", False,
                    "verifier property also supplied to model via " + " + ".join(_blocks))
        return "verifier-independent", True, "verifier property not supplied to model"
    if _tag == "semgrep_blind_unverified":
        return "none", False, "no oracle applied by the automated classifier"
    return "unknown", False, f"unrecognised reason tag: {_tag!r}"


_levels = (("l0", _l0_eval), ("l1", _l1c), ("l2", _l2c))
_rows = []
for _lay, _recs in _levels:
    for _r in _recs:
        if _r["outcome"] == "EXCLUDED":
            continue
        _orc, _indep, _why = _provenance(_r, _lay)
        _k = _key(_r)
        _fc, _hint = _prompt_context(_k, _r["vuln_dir"], _lay)
        _rows.append({
            "level":          _lay.upper(),
            "case":           _k,
            "module":         _r["vuln_dir"],
            "filename":       _r["filename"],
            "known_cwe":      _r.get("cwe_id", ""),
            "validated_label": _r["outcome"],
            "oracle":         _orc,
            "independent_of_prompt": _indep,
            "frontend_context_in_prompt": _fc,
            "fix_hint_in_prompt":  _hint,
            "smoke_test":     _smoke_status(_k, _lay),
            "detail":         _why,
        })

_ORDER = ["replay", "manual_review", "scanner", "target_weakness_mismatch",
          "verifier-independent", "verifier-conformity", "none", "unknown"]

# ── Table for the paper: provenance of the VALIDATED correct fixes ──────────
_cfr = [r for r in _rows if r["validated_label"] == "CFR"]
_by = defaultdict(lambda: defaultdict(int))
for _r in _cfr:
    _by[_r["oracle"]][_r["level"]] += 1

print("=" * 74)
print("ORACLE PROVENANCE OF THE VALIDATED CORRECT FIXES")
print("=" * 74)
print(f"  {'oracle':24}{'L0':>5}{'L1':>5}{'L2':>5}{'total':>8}   independent?")
print("  " + "-" * 70)
for _o in _ORDER:
    if _o not in _by:
        continue
    _d = _by[_o]
    _t = sum(_d.values())
    _ind = "yes" if _o in ("replay", "manual_review", "scanner",
                           "target_weakness_mismatch", "verifier-independent") else "NO"
    print(f"  {_o:24}{_d.get('L0',0):>5}{_d.get('L1',0):>5}{_d.get('L2',0):>5}{_t:>8}   {_ind}")
_tot = len(_cfr)
_ind_n = sum(1 for r in _cfr if r["independent_of_prompt"])
print("  " + "-" * 70)
print(f"  {'TOTAL':24}"
      f"{sum(1 for r in _cfr if r['level']=='L0'):>5}"
      f"{sum(1 for r in _cfr if r['level']=='L1'):>5}"
      f"{sum(1 for r in _cfr if r['level']=='L2'):>5}"
      f"{_tot:>8}")
print(f"\n  independent of the prompt : {_ind_n}/{_tot}")
print(f"  conformity or no oracle   : {_tot - _ind_n}/{_tot}")

_n_replay = sum(1 for r in _cfr if r["oracle"] == "replay")
_n_manual = sum(1 for r in _cfr if r["oracle"] == "manual_review")
_n_scan   = sum(1 for r in _cfr if r["oracle"] == "scanner")
_n_vind   = sum(1 for r in _cfr if r["oracle"] == "verifier-independent")
_n_vcon   = sum(1 for r in _cfr if r["oracle"] == "verifier-conformity")
_n_none   = sum(1 for r in _cfr if r["oracle"] == "none")
print("\n" + "=" * 74)
print("PASTE INTO SECTION 6.8.5")
print("=" * 74)
print(
    f"Of the {_tot} validated correct fixes, {_n_replay} were confirmed by exploit replay\n"
    f"against the running application, {_n_manual} by independent manual review against DVWA's\n"
    f"reference implementation, {_n_scan} by the disappearance of a Semgrep finding present in\n"
    f"the unpatched file, and {_n_vind} by a structural verifier whose property is not described\n"
    f"to the model. The remaining {_n_vcon + _n_none} rest on structural verifiers whose property is also\n"
    f"supplied to the model as application context or repair guidance ({_n_vcon}), or on no oracle\n"
    f"at all ({_n_none}), and therefore establish conformity to a specified repair pattern rather than\n"
    f"independent elimination of the weakness."
)

# ── Full per-case record for the replication package ───────────────────────
_prov_json = L2_RESULTS_DIR / "oracle_provenance.json"
_prov_csv  = L2_RESULTS_DIR / "oracle_provenance.csv"
_prov_json.write_text(json.dumps(_rows, indent=2))
with open(_prov_csv, "w", newline="") as _f:
    _w = csv.DictWriter(_f, fieldnames=list(_rows[0].keys()))
    _w.writeheader()
    _w.writerows(_rows)
print(f"\nsaved -> {_prov_json}")
print(f"saved -> {_prov_csv}   ({len(_rows)} rows)")

# ── Diagnostic: which modules rest wholly on conformity checks ──────────────
print("\n" + "=" * 74)
print("MODULES WHOSE CORRECT FIXES REST ONLY ON CONFORMITY OR NO ORACLE")
print("=" * 74)
_mod = defaultdict(list)
for _r in _cfr:
    _mod[_r["module"]].append(_r["oracle"])
for _m in sorted(_mod):
    if all(_o in ("verifier-conformity", "none") for _o in _mod[_m]):
        print(f"  {_m:18} {len(_mod[_m])} correct fix(es): {sorted(set(_mod[_m]))}")

# ── Diagnostic for report item C1: the bac / CWE-89 mapping ────────────────
print("\n" + "=" * 74)
print("DIAGNOSTIC — Broken Access Control (bac) classification basis")
_cv = globals().get("CUSTOM_VERIFIERS")          # defined in CELL 7; optional here
_cv_txt = ("bac" in _cv) if _cv is not None else "CELL 7 not run - check manually"
print(f"VULN_CWE_MAP['bac'] = {VULN_CWE_MAP.get('bac')!r}   "
      f"custom verifier registered: {_cv_txt}")
print("=" * 74)
for _r in _rows:
    if _r["module"] == "bac":
        print(f"  {_r['level']}  {_r['case']:22} label={_r['validated_label']:4} "
              f"oracle={_r['oracle']:22} known_cwe={_r['known_cwe']}")
print("  If known_cwe is 89 (SQL injection) but the module's weakness is session")
print("  fixation, the Semgrep comparison ran against the wrong target CWE.")